# Módulo 4 — Búsqueda híbrida


In [ ]:
!pip install sentence-transformers rank_bm25 -q


In [ ]:
from pathlib import Path
from sentence_transformers import SentenceTransformer, util
import numpy as np, re, requests, os, json

# Descarga del corpus desde GitHub si estás en Colab.
# Editá OWNER y REPO si cambiaste el nombre propuesto.
OWNER = "TU_USUARIO"
REPO = "curso-rag-acmecloud"
BASE = f"https://raw.githubusercontent.com/{OWNER}/{REPO}/main/corpus"

for nombre in ["catalogo_sistemas.txt","politicas_it.txt","manual_soporte.txt"]:
    r = requests.get(f"{BASE}/{nombre}", timeout=30)
    if r.status_code == 200:
        Path(f"/content/{nombre}").write_text(r.text, encoding="utf-8")
    else:
        print("No pude descargar", nombre, "- subilo manualmente o configurá OWNER/REPO.")


In [ ]:
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

archivos = ["/content/catalogo_sistemas.txt","/content/politicas_it.txt","/content/manual_soporte.txt"]
docs = {Path(p).name:Path(p).read_text(encoding="utf-8") for p in archivos}
def fichas(t):
    return ["###"+p.strip() for p in t.split("###")[1:] if p.strip()]
def prosa(t):
    return [p.strip() for p in t.split("\n\n") if p.strip()]
chunks=[]; origen=[]
for n,t in docs.items():
    ps=fichas(t) if "catalogo" in n else prosa(t)
    for p in ps: chunks.append(p); origen.append(n)
vectores=modelo.encode(chunks)

def tokenizar(t): return re.findall(r"\w+",t.lower())
bm25=BM25Okapi([tokenizar(c) for c in chunks])


In [ ]:
pregunta="¿Qué significa el error VPN-1047?"
vq=modelo.encode(pregunta)
scores_vec=util.cos_sim(vq,vectores)[0].numpy()
orden_vec=np.argsort(-scores_vec)[:8]
orden_bm=np.argsort(-bm25.get_scores(tokenizar(pregunta)))[:8]

print("VECTORIAL")
for i in orden_vec: print(round(float(scores_vec[i]),3), chunks[i][:90].replace("\n"," "))
print("\nBM25")
s_bm=bm25.get_scores(tokenizar(pregunta))
for i in orden_bm: print(round(float(s_bm[i]),3), chunks[i][:90].replace("\n"," "))


In [ ]:
def buscar_hibrido(pregunta,k=4,K=60):
    v=modelo.encode(pregunta)
    ov=np.argsort(-util.cos_sim(v,vectores)[0].numpy())
    ob=np.argsort(-bm25.get_scores(tokenizar(pregunta)))
    puntos={}
    for pos,idx in enumerate(ov,1): puntos[idx]=puntos.get(idx,0)+1/(K+pos)
    for pos,idx in enumerate(ob,1): puntos[idx]=puntos.get(idx,0)+1/(K+pos)
    return sorted(puntos,key=puntos.get,reverse=True)[:k]

for idx in buscar_hibrido(pregunta):
    print(origen[idx], chunks[idx][:220].replace("\n"," "))
